In [ ]:
import os
import pandas as pd
import commons as c
import numpy as np

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import plotly.graph_objects as go
import plotly.express as px

# Boxplots of mutant detectability (distance between mutant and original)


In [ ]:
csv_normal_path = 'results/dataframes/results_normal.csv'
df_normal = pd.read_csv(csv_normal_path, dtype=c.type_dict)
df_normal['nature'] = 'non-equivalent'

csv_equiv_path = 'results/dataframes/results_equiv.csv'
df_equiv = pd.read_csv(csv_equiv_path, dtype=c.type_dict)
df_equiv['nature'] = 'equivalent'
df = pd.concat([df_equiv, df_normal], ignore_index=True)

In [ ]:
df.head()

# Plot Acc/F1 as function of Threshold

In [ ]:
def find_best_thresholds(df, hw, m, start=0, end=1, steps=100, plot=True):
    thresholds = np.linspace(start, end, steps)
    f1_scores = []
    acc_scores = []

    best_thresh_f1 = None
    best_thresh_acc = None
    best_f1 = -1
    best_acc = -1

    # Convert 'equivalent' to 1 and 'non-equivalent' to 0
    y_true = df['nature'].map({'equivalent': 1, 'non-equivalent': 0}).values

    for t in thresholds:
        if m == 'F':
            y_pred = (df['noisy_distance'] >= t).astype(int)
        else:
            y_pred = (df['noisy_distance'] <= t).astype(int)
        f1 = f1_score(y_true, y_pred)
        acc = accuracy_score(y_true, y_pred)
        f1_scores.append(f1)
        acc_scores.append(acc)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh_f1 = t
        if acc > best_acc:
            best_acc = acc
            best_thresh_acc = t

    if plot:
        fig = go.Figure()
        colors = px.colors.qualitative.Prism

        fig.add_trace(go.Scatter(
            x=thresholds, y=acc_scores,
            mode='lines', name='Accuracy',
            line=dict(color=colors[6])
        ))

        fig.add_trace(go.Scatter(
            x=thresholds, y=f1_scores,
            mode='lines', name='F1 Score',
            line=dict(color=colors[7])
        ))

        fig.add_vline(
            x=best_thresh_acc,
            line_dash="dash",
            line_color=colors[6],
            annotation_text=f"Best Acc: {best_thresh_acc:.3f}",
            annotation_position="top left"
        )

        fig.add_vline(
            x=best_thresh_f1,
            line_dash="dash",
            line_color=colors[7],
            annotation_text=f"Best F1: {best_thresh_f1:.3f}",
            annotation_position="top right"
        )
        
        i = 4
        
        for t in c.thresholds:
            threshold_value = c.get_tolerance_values(hw, t)[c.metrics.get(m)]
            fig.add_vline(
                x=threshold_value,
                line_dash="dot",
                line_color=colors[i],
                annotation_text=f"{t}",
                annotation_position="top right"
            )
            i = i - 1

        fig.update_layout(
            xaxis_title="Threshold",
            yaxis_title="Score",
            xaxis_range=[start, end]
        )

        output_folder = 'results/test_thresholds/zoom/'
        file_name = f"{hw}_acc_f1_{m}_500_steps"
        c.setup_layout_and_save(fig, output_folder, file_name, yaxis_range=[0, 1], height=700)

    return best_thresh_f1, best_f1, best_thresh_acc, best_acc


In [ ]:
metrics = {'H': (0.05,0.42), 'J': (0.05,0.37), 'T': (0,0.22), 'F': (0.5,1), 'E': (0.012,0.6)}
for hw in c.hardware:
    print(f"===================== {hw} =======================")
    df_hw = df[df['hardware'] == hw]
    for m, (start,end) in metrics.items():
        #print(f"===================== {m} =======================")
        df_metric = df_hw[df_hw['metric'] == m]
        best_thresh_f1, best_f1, best_thresh_acc, best_acc = find_best_thresholds(df_metric, hw, m, start=start, end=end, steps=500)
        print(f"Metric: {m}, Best threshold (according to Accuracy): {best_thresh_acc:.4f}, Best threshold (according to F1 score): {best_thresh_f1:.4f}")
        #print(f"Metric: {m}, Best threshold: {best_thresh_acc:.4f}, Best Accuracy: {best_acc:.4f}")
        #print(f"Metric: {m}, Best threshold: {best_thresh_f1:.4f}, Best F1 score: {best_f1:.4f}")